In [17]:
import json
from pyexpat.errors import messages

import openai
from openai.types.chat import ChatCompletionMessage
import requests

client = openai.OpenAI()
api_url = 'https://nomad-movies.nomadcoders.workers.dev'
messages = []


def get_popular_movies():
    response = requests.get(f'{api_url}/movies')
    response.raise_for_status()
    return response.text  # 또는 response.json()


def get_movie_details(id):
    response = requests.get(f'{api_url}/movies/{id}')
    response.raise_for_status()
    return response.text


def get_movie_credits(id):
    response = requests.get(f'{api_url}/movies/{id}/credits')
    response.raise_for_status()
    return response.text


def get_similar_movies(id):
    response = requests.get(f'{api_url}/movies/{id}/similar')
    response.raise_for_status()
    return response.text


FUNCTION_MAP = {
    "get_popular_movies": get_popular_movies,
    "get_movie_details": get_movie_details,
    "get_movie_credits": get_movie_credits,
    "get_similar_movies": get_similar_movies,
}

TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_popular_movies",
            "description": "Get a list of popular movies"
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_details",
            "description": "Get a movie's details by ID",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {
                        "description": "Movie ID",
                        "type": "integer"
                    }
                },
                "required": ["id"]
            }
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_credits",
            "description": "Get a movie's cast and crew by ID",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {
                        "description": "Movie ID",
                        "type": "integer"
                    }
                },
                "required": ["id"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_similar_movies",
            "description": "Get a list of similar movies",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {
                        "description": "Movie ID",
                        "type": "integer"
                    }
                },
                "required": ["id"]
            }
        }
    }
]

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "Hello!"}],
    tools=TOOLS
)



In [18]:
def process_ai_message(ai_message: ChatCompletionMessage):
    if ai_message.tool_calls:
        for tool_call in ai_message.tool_calls:
            function_name = tool_call.function.name
            function_to_call = FUNCTION_MAP[function_name]
            arguments = tool_call.function.arguments
            print(
                f"Agent: [{function_name} with arguments: {arguments} 호출]")

            try:
                arguments = json.loads(arguments)
            except json.JSONDecodeError:
                arguments = {}

            if arguments == {}:
                result = function_to_call()
            else:
                result = function_to_call(**arguments)

            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "name": function_name,
                "content": json.dumps(result, ensure_ascii=False),
            })

        call_ai()
    else:
        messages.append({"role": "assistant", "content": ai_message.content})
        print(f"AI: {ai_message.content}")


def call_ai():
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        tools=TOOLS
    )
    ai_message = response.choices[0].message
    assistant_message = {
        "role": "assistant",
        "content": ai_message.content or "",
    }
    if ai_message.tool_calls:
        assistant_message['tool_calls'] = [
            {
                "id": tc.id,
                "type": tc.type,
                "function": {
                    "name": tc.function.name,
                    "arguments": tc.function.arguments,
                },
            }
            for tc in ai_message.tool_calls
        ]
    messages.append(assistant_message)

    if ai_message.content:
        print(f"AI: {ai_message.content}")
    process_ai_message(response.choices[0].message)


while True:
    message = input("Send a message to the LLM").strip()
    if message == "exit" or message == "quit" or message == "bye":
        print("bye.")
        break
    else:
        messages.append({
            "role": "user",
            "content": message
        })
        print(f"You: {messages[-1]['content']}")
        call_ai()



You: 지금 인기 있는 영화 알려줘
Agent: [get_popular_movies with arguments: {} 호출]
AI: 지금 인기 있는 영화는 다음과 같습니다:

1. **Shelter**
   - 개봉일: 2026-01-28
   - 평점: 6.94
   - 개요: 외딴 섬에서 자발적 망명생활을 하는 남자가 폭풍에서 어린 소녀를 구하면서 자신의 과거와 관련된 적들로부터 그녀를 보호하기 위해 고군분투하는 이야기입니다.
   - ![Shelter](https://image.tmdb.org/t/p/w780/buPFnHZ3xQy6vZEHxbHgL1Pc6CR.jpg)

2. **Mercy**
   - 개봉일: 2026-01-20
   - 평점: 7.12
   - 개요: 가까운 미래, 자신의 아내를 살해한 혐의로 재판 중인 형사가 90분 안에 AI 판사에게 자신의 무죄를 증명해야 하는 이야기입니다.
   - ![Mercy](https://image.tmdb.org/t/p/w780/pyok1kZJCfyuFapYXzHcy7BLlQa.jpg)

3. **Les Orphelins (The Orphans)**
   - 개봉일: 2025-08-20
   - 평점: 6.10
   - 개요: 과거의 친구들이 서로의 다른 삶 속에서 만나고, 첫사랑의 딸이 복수를 위해 나서는 이야기입니다.
   - ![The Orphans](https://image.tmdb.org/t/p/w780/hP7mjZr2SVfjAorlRHTdV1XZmHY.jpg)

4. **The Bluff**
   - 개봉일: 2026-02-17
   - 평점: 5.98
   - 개요: 한 외딴 섬에서 평화롭게 지내던 전직 해적이 복수심에 불타는 전 선장과 맞서 싸우는 이야기입니다.
   - ![The Bluff](https://image.tmdb.org/t/p/w780/sojEzvfxR2DBcDSJyAisX8TWjov.jpg)

5. **A Woman Scorned**
   - 개봉일: 2025-06-09
 